# 05 — Rank-difference deconvolution

Merge Yan 2020 DMS with evolutionary scores and SPURS. Deconvolution uses `-spurs` (stabilizing is higher) and DMS `stability` as a second proxy. ThermoMPNN / RSA analog are out of scope.


In [ ]:
from pathlib import Path

import pandas as pd

from dualde.deconvolution import deconvolve_evo_vs_stability, merge_evo_spurs_dms
from dualde.paths import example_dir

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

evo_path = OUT / "evolutionary.csv"
if not evo_path.is_file():
    evo_path = EXAMPLE / "scores" / "evolutionary.csv"
spurs_path = OUT / "5lpg_spurs.csv"
if not spurs_path.is_file():
    spurs_path = EXAMPLE / "scores" / "5lpg_spurs.csv"

evo = pd.read_csv(evo_path)
spurs = pd.read_csv(spurs_path)
activity = pd.read_csv(EXAMPLE / "dms" / "yan2020_activity.csv")
stability = pd.read_csv(EXAMPLE / "dms" / "yan2020_stability.csv")

merged = merge_evo_spurs_dms(evo, spurs, activity, stability)
print("merged", merged.shape)

evo_cols = [c for c in ("EVmutation", "EVmutation2", "EVE", "ESM2_35M", "ESM2_650M", "ESM2_3B") if c in merged.columns]
merged = deconvolve_evo_vs_stability(merged, evo_cols=evo_cols, stability_cols=("neg_spurs", "stability"))
print([c for c in merged.columns if c.endswith("_model")])
merged.to_csv(OUT / "deconvolution.csv", index=False)
merged.head()
